# LeetCode 1612 — Check If Two Expression Trees are Equivalent bằng Python

Link đề: https://leetcode.com/problems/check-if-two-expression-trees-are-equivalent/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Hash Table, Binary Tree, Counting

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class Node(object):
#     def __init__(self, val=" ", left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def checkEquivalence(self, root1: 'Node', root2: 'Node') -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu **cây biểu thức nhị phân** (binary expression tree): lá là biến, nút trong là toán tử, và cách dựng cây từ mảng level-order kiểu LeetCode.
2. Biến câu hỏi "hai biểu thức luôn bằng nhau với mọi giá trị biến" thành câu hỏi **đếm**: nhờ tính giao hoán và kết hợp của phép `+`, chỉ cần so sánh **số lần xuất hiện của mỗi chữ cái**.
3. Viết và so sánh nhiều lời giải: thử gán giá trị (one-hot), sắp xếp lá, `Counter` cộng/trừ bằng DFS đệ quy, DFS **không đệ quy** bằng stack.
4. Nhận ra bẫy **đệ quy quá sâu** trong Python (giới hạn mặc định 1000) khi cây lệch có tới 4999 nút.
5. Giải **follow-up** có thêm phép `-` bằng **vector hệ số** (mỗi biến có hệ số nguyên, có thể âm).

## 2. Đề bài & ví dụ

**Cây biểu thức nhị phân** là cây nhị phân mà mỗi nút có **0 hoặc 2 con**:
- nút lá (0 con) chứa một **biến** — một chữ cái thường `'a'..'z'`;
- nút trong (2 con) chứa một **toán tử**. Trong bài này chỉ có toán tử `'+'`.

Cho gốc của hai cây `root1`, `root2`. Trả về `True` nếu hai cây **tương đương**, ngược lại `False`.
Hai cây tương đương nếu chúng **cho cùng một giá trị bất kể các biến được gán giá trị gì**.

**Ví dụ 1:** `root1 = [x]`, `root2 = [x]` → `True` (cả hai đều là biểu thức `x`).

**Ví dụ 2:** `root1 = [+,a,+,null,null,b,c]`, `root2 = [+,+,a,b,c]` → `True`.
Cây 1 là `a + (b + c)`, cây 2 là `(b + c) + a` — cùng giá trị với mọi `a, b, c`.

**Ví dụ 3:** `root1 = [+,a,+,null,null,b,c]`, `root2 = [+,+,a,b,d]` → `False`.
Cây 1 là `a + (b + c)`, cây 2 là `(b + d) + a`; chọn `c = 1, d = 0` là thấy khác.

> Lưu ý đọc mảng: mảng được ghi theo **level-order**, `null` là vị trí trống. Trong `[+,+,a,b,c]`: gốc `+`, con trái `+`, con phải `a`; hai con của nút `+` bên trái là `b`, `c`. Vậy cây 2 là `(b + c) + a`.

**Ràng buộc:**
- Số nút của hai cây **bằng nhau**, là số **lẻ**, nằm trong `[1, 4999]`.
- `Node.val` là `'+'` hoặc một chữ cái thường.
- Đảm bảo cây là cây biểu thức nhị phân hợp lệ.

**Follow-up:** Nếu cây có thêm toán tử `'-'` (phép trừ) thì lời giải phải thay đổi thế nào?

In [1]:
import random
import time
import sys
from collections import Counter, deque
from typing import Optional, List

class Node(object):
    """Lớp nút giống LeetCode: val mặc định là một dấu cách."""
    def __init__(self, val=" ", left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = null)."""
    if not arr or arr[0] is None:
        return None
    root = Node(arr[0]); q = deque([root]); i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = Node(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = Node(arr[i]); q.append(node.right)
        i += 1
    return root

def to_list(root):
    """Xuất cây ra mảng level-order kiểu LeetCode (bỏ các None thừa ở cuối)."""
    out, q = [], deque([root])
    while q:
        x = q.popleft()
        if x is None:
            out.append(None); continue
        out.append(x.val); q.append(x.left); q.append(x.right)
    while out and out[-1] is None:
        out.pop()
    return out

examples = [
    (["x"], ["x"], True),
    (["+", "a", "+", None, None, "b", "c"], ["+", "+", "a", "b", "c"], True),
    (["+", "a", "+", None, None, "b", "c"], ["+", "+", "a", "b", "d"], False),
]
for a1, a2, exp in examples:
    r1, r2 = build_tree(a1), build_tree(a2)
    print(to_list(r1), "|", to_list(r2), "| kỳ vọng:", exp)

['x'] | ['x'] | kỳ vọng: True
['+', 'a', '+', None, None, 'b', 'c'] | ['+', '+', 'a', 'b', 'c'] | kỳ vọng: True
['+', 'a', '+', None, None, 'b', 'c'] | ['+', '+', 'a', 'b', 'd'] | kỳ vọng: False


Ta viết thêm hàm **đệ quy** `expr` để in cây thành biểu thức có ngoặc (chỉ dùng cho cây nhỏ khi minh họa — cây sâu sẽ được xử lý bằng stack ở các mục sau), và kiểm tra lại rằng `to_list(build_tree(arr)) == arr` (dựng rồi xuất phải ra đúng mảng ban đầu).

In [2]:
def expr(root):
    """In biểu thức có ngoặc — đệ quy, chỉ dùng cho cây nhỏ."""
    if root.left is None and root.right is None:
        return root.val
    return "(" + expr(root.left) + " " + root.val + " " + expr(root.right) + ")"

for a1, a2, exp in examples:
    r1, r2 = build_tree(a1), build_tree(a2)
    assert to_list(r1) == a1 and to_list(r2) == a2
    print(f"{expr(r1):<15} vs {expr(r2):<15} -> kỳ vọng {exp}")

x               vs x               -> kỳ vọng True
(a + (b + c))   vs ((b + c) + a)   -> kỳ vọng True
(a + (b + c))   vs ((b + d) + a)   -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Cây biểu thức và cách "đọc" giá trị của nó

Giá trị của một nút:
- nếu là lá chữ `v` → giá trị là `env[v]` (giá trị đang gán cho biến `v`);
- nếu là `+` → giá trị = giá trị cây con trái + giá trị cây con phải.

Đây chính là **duyệt hậu thứ tự** (post-order): tính hai con trước, rồi mới tính nút cha.

In [3]:
def evaluate(root, env):
    """Tính giá trị cây với bảng gán env: dict chữ cái -> số."""
    if root.left is None:            # lá (cây hợp lệ: 0 hoặc 2 con)
        return env[root.val]
    a = evaluate(root.left, env)
    b = evaluate(root.right, env)
    return a + b                     # bài gốc chỉ có '+'

r = build_tree(["+", "a", "+", None, None, "b", "c"])
print(expr(r), "với a=1,b=2,c=3 ->", evaluate(r, {"a": 1, "b": 2, "c": 3}))

(a + (b + c)) với a=1,b=2,c=3 -> 6


### 3.2 `collections.Counter` — đếm số lần xuất hiện

`Counter` là một `dict` đặc biệt: truy cập khóa chưa có trả về `0` (không lỗi), nên viết `cnt[k] += 1` rất gọn. Hai `Counter` so sánh bằng `==` theo từng khóa; từ Python 3.10, các khóa có giá trị `0` được coi như không có khi so sánh.

In [4]:
c1 = Counter("abca")
c2 = Counter(["a", "c", "b", "a"])
print(c1, c2, c1 == c2)
c3 = Counter()
c3["z"] += 1
c3["z"] -= 1
print(c3, "| c3 == Counter():", c3 == Counter(), "| all zero:", all(v == 0 for v in c3.values()))

Counter({'a': 2, 'b': 1, 'c': 1}) Counter({'a': 2, 'c': 1, 'b': 1}) True
Counter({'z': 0}) | c3 == Counter(): True | all zero: True


### 3.3 Mảng 26 phần tử thay cho dict

Vì biến chỉ là chữ thường, ta có thể dùng `cnt = [0] * 26` và chỉ số `ord(ch) - ord('a')`. Cách này nhanh và dễ chuyển sang C++/Java.

In [5]:
cnt = [0] * 26
for ch in "banana":
    cnt[ord(ch) - ord("a")] += 1
print({chr(i + 97): cnt[i] for i in range(26) if cnt[i]})

{'a': 3, 'b': 1, 'n': 2}


### 3.4 Duyệt cây không đệ quy bằng stack

Python giới hạn độ sâu đệ quy mặc định là `1000`. Cây 4999 nút có thể là một "chuỗi" lệch sâu ~2500 tầng ⇒ đệ quy có thể gây `RecursionError`. Duyệt bằng **stack** (list) thì không có giới hạn này: lấy một nút ra, xử lý, đẩy các con vào.

In [6]:
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def leaves_iter(root):
    """Trả về danh sách giá trị các lá, duyệt bằng stack (không đệ quy)."""
    out, st = [], [root]
    while st:
        x = st.pop()
        if x.left is None:
            out.append(x.val)
        else:
            st.append(x.right); st.append(x.left)   # đẩy phải trước để lấy trái trước
    return out

print(leaves_iter(build_tree(["+", "+", "a", "b", "c"])))

Giới hạn đệ quy hiện tại: 1000
['b', 'c', 'a']


### 3.5 Tính chất giao hoán và kết hợp của `+`

- **Giao hoán:** `x + y = y + x` ⇒ được **đổi chỗ** hai con của mọi nút `+`.
- **Kết hợp:** `(x + y) + z = x + (y + z)` ⇒ được **đổi cách đặt ngoặc**.

Kết hợp hai tính chất, mọi biểu thức chỉ có `+` đều bằng **tổng của tất cả các lá**, không phụ thuộc hình dạng cây. Đây là chìa khóa của bài.

In [7]:
x, y, z = 7, -3, 10
print((x + y) + z == x + (y + z), x + y == y + x, (z + x) + y == x + (y + z))

True True True


## 4. Ý tưởng nền tảng / trực giác

Tính bằng tay trên **Ví dụ 2**:
- Cây 1: `a + (b + c)` → khi "phá ngoặc" được `a + b + c`. Các lá: `a, b, c`.
- Cây 2: `(b + c) + a` → `b + c + a`. Các lá: `b, c, a`.

Cùng một **túi** (multiset) lá `{a:1, b:1, c:1}` ⇒ tương đương.

**Ví dụ 3:** cây 1 có túi `{a:1, b:1, c:1}`, cây 2 có túi `{a:1, b:1, d:1}`. Khác nhau ở `c` và `d` ⇒ chọn `c=1`, mọi biến khác `0`: cây 1 cho `1`, cây 2 cho `0` ⇒ không tương đương.

Một ví dụ tự dựng (không có trong đề — đã kiểm chứng bằng brute force ở mục 5, nên đối chiếu lại đề gốc): `a + (a + b)` và `(b + b) + a`. Tập chữ cái giống nhau `{a, b}` nhưng **số lần** khác: `{a:2, b:1}` vs `{a:1, b:2}` ⇒ `False`. Vậy phải đếm **số lần**, không chỉ xem có/không.

Biểu thức chỉ có `+` luôn có dạng **tuyến tính**: `giá trị = c_a·a + c_b·b + … + c_z·z`, với `c_v` là số lần lá `v` xuất hiện. Hai đa thức tuyến tính bằng nhau với mọi giá trị biến ⇔ **mọi hệ số bằng nhau**.

In [8]:
def bag(arr):
    return Counter(v for v in arr if v is not None and v != "+")

for a1, a2, exp in examples + [(["+", "a", "+", None, None, "a", "b"], ["+", "+", "a", "b", "b"], False)]:
    print(f"{expr(build_tree(a1)):<14} túi={dict(bag(a1))}   {expr(build_tree(a2)):<14} túi={dict(bag(a2))}")

x              túi={'x': 1}   x              túi={'x': 1}
(a + (b + c))  túi={'a': 1, 'b': 1, 'c': 1}   ((b + c) + a)  túi={'a': 1, 'b': 1, 'c': 1}
(a + (b + c))  túi={'a': 1, 'b': 1, 'c': 1}   ((b + d) + a)  túi={'a': 1, 'b': 1, 'd': 1}
(a + (a + b))  túi={'a': 2, 'b': 1}   ((b + b) + a)  túi={'a': 1, 'b': 2}


## 5. Cách 1 — Brute Force / baseline: thử gán giá trị theo kiểu one-hot

**Ý tưởng (bám sát định nghĩa):** "tương đương" nghĩa là bằng nhau với **mọi** phép gán. Không thể thử vô hạn phép gán, nhưng vì biểu thức tuyến tính `Σ c_v·v`, chỉ cần thử **26 phép gán one-hot**: lần thứ `i` gán chữ thứ `i` bằng `1`, mọi chữ khác bằng `0`. Khi đó giá trị cây đúng bằng hệ số `c_i`. Nếu cả 26 lần hai cây cho cùng giá trị thì mọi hệ số bằng nhau ⇒ tương đương.

(Một baseline khác hay gặp là gán **ngẫu nhiên** rồi so sánh nhiều lần — đúng với xác suất rất cao nhưng không chắc chắn tuyệt đối; cách one-hot là **tất định**.)

In [9]:
def evaluate_iter(root, env):
    """Tính giá trị bằng stack (post-order), tránh đệ quy sâu."""
    st, vals = [(root, False)], []
    while st:
        x, done = st.pop()
        if x.left is None:
            vals.append(env.get(x.val, 0))
        elif done:
            b = vals.pop(); a = vals.pop()
            vals.append(a + b)
        else:
            st.append((x, True)); st.append((x.right, False)); st.append((x.left, False))
    return vals[0]

def check_bruteforce(root1, root2):
    for i in range(26):
        env = {chr(97 + i): 1}            # one-hot: chữ thứ i = 1, còn lại = 0
        if evaluate_iter(root1, env) != evaluate_iter(root2, env):
            return False
    return True

for a1, a2, exp in examples:
    got = check_bruteforce(build_tree(a1), build_tree(a2))
    print(got, "kỳ vọng", exp)
    assert got == exp

True kỳ vọng True
True kỳ vọng True
False kỳ vọng False


Kiểm chứng ví dụ tự dựng `a + (a + b)` vs `(b + b) + a` (ghi chú: ví dụ tự dựng, đối chiếu lại đề gốc):

In [10]:
mine1 = ["+", "a", "+", None, None, "a", "b"]
mine2 = ["+", "+", "a", "b", "b"]
print(expr(build_tree(mine1)), "vs", expr(build_tree(mine2)), "->", check_bruteforce(build_tree(mine1), build_tree(mine2)))

(a + (a + b)) vs ((b + b) + a) -> False


**Độ phức tạp:** mỗi lần đánh giá `O(n)`, làm 26 lần ⇒ `O(26·n) = O(n)` thời gian, `O(n)` bộ nhớ cho stack.

**Nút thắt:** hằng số 26 khá lớn (đi qua mỗi cây 26 lần), và cách này **phụ thuộc vào việc biểu thức tuyến tính** — nếu có phép nhân, one-hot không còn đủ. Nhưng nó dạy ta điều quan trọng: thứ thực sự cần so sánh là **hệ số của từng biến**. Vậy sao không đếm hệ số trực tiếp trong **một lần duyệt**?

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với chỉ phép `+`, giá trị cây = tổng các lá ⇒ hệ số của biến `v` = **số lá mang chữ `v`**.

**Quy tắc:**
- Nếu với mọi chữ `v`, `count1[v] == count2[v]` **thì** hai cây tương đương.
- Nếu tồn tại `v` với `count1[v] != count2[v]` **thì** gán `v = 1`, các biến khác `= 0` cho ra hai giá trị khác nhau ⇒ **không** tương đương.

**Mẹo một bộ đếm:** duyệt cây 1 thì `cnt[v] += 1`, duyệt cây 2 thì `cnt[v] -= 1`. Cuối cùng tương đương ⇔ **mọi** giá trị trong `cnt` bằng `0`.

**Invariant khi duyệt:** sau khi đã thăm một tập lá `L1` của cây 1 và `L2` của cây 2, `cnt[v] = #(v trong L1) − #(v trong L2)`.

**Bỏ qua nút `+`:** nút `+` không đóng góp hệ số nào, chỉ cần đi xuống hai con.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thu thập lá rồi sắp xếp

**Giải thích từng đoạn:**
1. Duyệt mỗi cây, gom mọi chữ ở lá vào một list.
2. Sắp xếp hai list. Hai multiset bằng nhau ⇔ hai list đã sắp xếp bằng nhau.

Dễ nghĩ ra, đúng, nhưng tốn `O(n log n)` do sắp xếp.

In [11]:
def check_sort(root1, root2):
    return sorted(leaves_iter(root1)) == sorted(leaves_iter(root2))

for a1, a2, exp in examples:
    r1, r2 = build_tree(a1), build_tree(a2)
    print(sorted(leaves_iter(r1)), sorted(leaves_iter(r2)), "->", check_sort(r1, r2))
    assert check_sort(r1, r2) == exp

['x'] ['x'] -> True
['a', 'b', 'c'] ['a', 'b', 'c'] -> True
['a', 'b', 'c'] ['a', 'b', 'd'] -> False


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionSort:
    def checkEquivalence(self, root1: 'Node', root2: 'Node') -> bool:
        def leaves(root):
            out, st = [], [root]
            while st:
                x = st.pop()
                if x.val == '+':
                    st.append(x.left); st.append(x.right)
                else:
                    out.append(x.val)
            return out
        return sorted(leaves(root1)) == sorted(leaves(root2))

print([SolutionSort().checkEquivalence(build_tree(a), build_tree(b)) for a, b, _ in examples])

[True, True, False]


**Độ phức tạp Cách 2:** Time `O(n log n)` (sắp xếp `(n+1)/2` lá), Space `O(n)`.

### 7.2 Cách 3 — DFS đệ quy với một `Counter` cộng/trừ

**Giải thích từng đoạn:**
1. `dfs(node, sign)`: nếu `node` là lá chữ → `cnt[node.val] += sign`; nếu là `+` → gọi tiếp hai con.
2. Gọi `dfs(root1, +1)` rồi `dfs(root2, -1)`.
3. Trả về `all(v == 0 for v in cnt.values())`.

Code demo trước, có in `cnt` sau mỗi cây:

In [13]:
def check_counter_demo(root1, root2):
    cnt = Counter()
    def dfs(node, sign):
        if node is None:
            return
        if node.val != "+":
            cnt[node.val] += sign
        dfs(node.left, sign)
        dfs(node.right, sign)
    dfs(root1, +1); print("  sau cây 1:", dict(cnt))
    dfs(root2, -1); print("  sau cây 2:", dict(cnt))
    return all(v == 0 for v in cnt.values())

for a1, a2, exp in examples:
    print(a1, "vs", a2)
    print("  =>", check_counter_demo(build_tree(a1), build_tree(a2)))

['x'] vs ['x']
  sau cây 1: {'x': 1}
  sau cây 2: {'x': 0}
  => True
['+', 'a', '+', None, None, 'b', 'c'] vs ['+', '+', 'a', 'b', 'c']
  sau cây 1: {'a': 1, 'b': 1, 'c': 1}
  sau cây 2: {'a': 0, 'b': 0, 'c': 0}
  => True
['+', 'a', '+', None, None, 'b', 'c'] vs ['+', '+', 'a', 'b', 'd']
  sau cây 1: {'a': 1, 'b': 1, 'c': 1}
  sau cây 2: {'a': 0, 'b': 0, 'c': 1, 'd': -1}
  => False


**Code nộp LeetCode (Cách 3):** thêm `sys.setrecursionlimit` để an toàn với cây lệch sâu ~2500.

In [14]:
class SolutionRecursive:
    def checkEquivalence(self, root1: 'Node', root2: 'Node') -> bool:
        sys.setrecursionlimit(10000)
        cnt = Counter()
        def dfs(node, sign):
            if node is None:
                return
            if node.val != '+':
                cnt[node.val] += sign
            dfs(node.left, sign)
            dfs(node.right, sign)
        dfs(root1, 1)
        dfs(root2, -1)
        return all(v == 0 for v in cnt.values())

print([SolutionRecursive().checkEquivalence(build_tree(a), build_tree(b)) for a, b, _ in examples])

[True, True, False]


**Độ phức tạp Cách 3:** Time `O(n)` (mỗi nút thăm 1 lần), Space `O(h)` cho ngăn xếp đệ quy (`h` = chiều cao, tệ nhất `~n/2`) + `O(26)` cho bộ đếm.

### 7.3 Cách 4 — DFS bằng stack + mảng 26 (tối ưu, không lo đệ quy)

**Giải thích từng đoạn:**
1. Đẩy `(root1, +1)` và `(root2, -1)` vào **cùng một stack** — dấu đi kèm từng nút.
2. Lấy ra một cặp `(x, s)`: nếu `x.val == '+'` thì đẩy `(x.left, s)` và `(x.right, s)`; ngược lại `cnt[ord(x.val) - 97] += s`.
3. Cuối cùng kiểm tra `not any(cnt)` (mọi phần tử bằng 0).

In [15]:
def check_stack(root1, root2):
    cnt = [0] * 26
    st = [(root1, 1), (root2, -1)]
    while st:
        x, s = st.pop()
        if x.val == "+":
            st.append((x.left, s)); st.append((x.right, s))
        else:
            cnt[ord(x.val) - 97] += s
    return not any(cnt)

print([check_stack(build_tree(a), build_tree(b)) for a, b, _ in examples])

[True, True, False]


**Code nộp LeetCode (Cách 4):** xem mục 9 (đây là bản được chọn để nộp).

**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(h)` cho stack (tệ nhất `O(n)`), bộ đếm `O(26)`.

### 7.4 Cách 5 — Follow-up: vector hệ số khi có thêm phép `-`

Với `-`, cây không còn là "tổng các lá": `a - b ≠ b - a`. Nhưng biểu thức vẫn **tuyến tính**: mỗi biến có hệ số nguyên (có thể âm).

**Giải thích từng đoạn:**
1. Mỗi nút trả về một vector 26 hệ số.
2. Lá chữ `v` → vector có `1` tại vị trí `v`.
3. Nút `+` → `trái + phải`; nút `-` → `trái − phải` (đổi dấu toàn bộ vector con phải).
4. Hai cây tương đương ⇔ hai vector bằng nhau.

Cách cài không đệ quy: đẩy **dấu tích lũy** xuống: con trái giữ dấu cha, con phải của nút `-` nhận dấu ngược lại. Lá chữ `v` cộng `dấu` vào `cnt[v]`. Đây là tổng quát hóa Cách 4.

In [16]:
def coef_vector(root):
    """Vector hệ số (dict) của biểu thức có + và -, duyệt bằng stack với dấu tích lũy."""
    cnt = Counter()
    st = [(root, 1)]
    while st:
        x, s = st.pop()
        if x.val == "+":
            st.append((x.left, s)); st.append((x.right, s))
        elif x.val == "-":
            st.append((x.left, s)); st.append((x.right, -s))
        else:
            cnt[x.val] += s
    return cnt

def check_with_minus(root1, root2):
    return coef_vector(root1) == coef_vector(root2)

t1 = build_tree(["-", "a", "-", None, None, "b", "c"])     # a - (b - c) = a - b + c
t2 = build_tree(["+", "-", "c", "a", "b"])                 # (a - b) + c
t3 = build_tree(["-", "-", "c", "a", "b"])                 # (a - b) - c
print(expr(t1), dict(coef_vector(t1)))
print(expr(t2), dict(coef_vector(t2)))
print(expr(t3), dict(coef_vector(t3)))
print(check_with_minus(t1, t2), check_with_minus(t1, t3))

(a - (b - c)) {'c': 1, 'b': -1, 'a': 1}
((a - b) + c) {'c': 1, 'b': -1, 'a': 1}
((a - b) - c) {'c': -1, 'b': -1, 'a': 1}
True False


Ví dụ `-` ở trên là **tự dựng** (đề gốc không có ví dụ cho follow-up) — ta kiểm chứng bằng cách gán ngẫu nhiên giá trị nhiều lần. Ghi chú: đối chiếu lại đề gốc.

In [17]:
def evaluate_pm(root, env):
    if root.left is None:
        return env[root.val]
    a, b = evaluate_pm(root.left, env), evaluate_pm(root.right, env)
    return a + b if root.val == "+" else a - b

rng = random.Random(1612)
for A, B in [(t1, t2), (t1, t3)]:
    same = all(evaluate_pm(A, e) == evaluate_pm(B, e)
               for e in ({ch: rng.randint(-50, 50) for ch in "abc"} for _ in range(200)))
    print("ngẫu nhiên 200 lần giống nhau:", same, "| vector:", check_with_minus(A, B))

ngẫu nhiên 200 lần giống nhau: True | vector: True
ngẫu nhiên 200 lần giống nhau: False | vector: False


**Code nộp (nếu đề có `-`):**

```python
class Solution:
    def checkEquivalence(self, root1: 'Node', root2: 'Node') -> bool:
        cnt = [0] * 26
        st = [(root1, 1), (root2, -1)]
        while st:
            x, s = st.pop()
            if x.val == '+':
                st.append((x.left, s)); st.append((x.right, s))
            elif x.val == '-':
                st.append((x.left, s)); st.append((x.right, -s))
            else:
                cnt[ord(x.val) - 97] += s
        return not any(cnt)
```

**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(h)` — giống Cách 4, chỉ thêm một nhánh `elif`.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 3**: `root1 = a + (b + c)`, `root2 = (b + d) + a`.

| bước | lấy ra (nút, dấu) | hành động | cnt khác 0 |
|---|---|---|---|
| 1 | (`+` gốc cây 2, −1) | đẩy hai con | {} |
| 2 | (`a`, −1) | cnt[a] −= 1 | {a:−1} |
| 3 | (`+`, −1) | đẩy `b`, `d` | {a:−1} |
| 4 | (`d`, −1) | cnt[d] −= 1 | {a:−1, d:−1} |
| 5 | (`b`, −1) | cnt[b] −= 1 | {a:−1, b:−1, d:−1} |
| 6 | (`+` gốc cây 1, +1) | đẩy hai con | … |
| … | … | … | cuối: {c:+1, d:−1} |

Stack là LIFO nên cây 2 (đẩy sau) được xử lý trước — thứ tự không quan trọng vì phép cộng giao hoán. Hàm `trace_stack` in đúng trạng thái từng bước:

In [18]:
def trace_stack(root1, root2):
    cnt = Counter()
    st = [(root1, 1, "T1"), (root2, -1, "T2")]
    step = 0
    while st:
        x, s, tag = st.pop()
        step += 1
        if x.val == "+":
            st.append((x.left, s, tag)); st.append((x.right, s, tag))
            act = "đẩy 2 con"
        else:
            cnt[x.val] += s
            act = f"cnt[{x.val}] {'+' if s > 0 else '-'}= 1"
        nz = {k: v for k, v in sorted(cnt.items()) if v}
        print(f"bước {step:>2} | {tag} nút '{x.val}' dấu {s:+d} | {act:<12} | stack={len(st)} | cnt≠0: {nz}")
    ans = all(v == 0 for v in cnt.values())
    print("Kết quả:", ans)
    return ans

trace_stack(build_tree(examples[2][0]), build_tree(examples[2][1]))

bước  1 | T2 nút '+' dấu -1 | đẩy 2 con    | stack=3 | cnt≠0: {}
bước  2 | T2 nút 'a' dấu -1 | cnt[a] -= 1  | stack=2 | cnt≠0: {'a': -1}
bước  3 | T2 nút '+' dấu -1 | đẩy 2 con    | stack=3 | cnt≠0: {'a': -1}
bước  4 | T2 nút 'd' dấu -1 | cnt[d] -= 1  | stack=2 | cnt≠0: {'a': -1, 'd': -1}
bước  5 | T2 nút 'b' dấu -1 | cnt[b] -= 1  | stack=1 | cnt≠0: {'a': -1, 'b': -1, 'd': -1}
bước  6 | T1 nút '+' dấu +1 | đẩy 2 con    | stack=2 | cnt≠0: {'a': -1, 'b': -1, 'd': -1}
bước  7 | T1 nút '+' dấu +1 | đẩy 2 con    | stack=3 | cnt≠0: {'a': -1, 'b': -1, 'd': -1}
bước  8 | T1 nút 'c' dấu +1 | cnt[c] += 1  | stack=2 | cnt≠0: {'a': -1, 'b': -1, 'c': 1, 'd': -1}
bước  9 | T1 nút 'b' dấu +1 | cnt[b] += 1  | stack=1 | cnt≠0: {'a': -1, 'c': 1, 'd': -1}
bước 10 | T1 nút 'a' dấu +1 | cnt[a] += 1  | stack=0 | cnt≠0: {'c': 1, 'd': -1}
Kết quả: False


False

Trace trên **Ví dụ 2** (kết quả phải là `True` — mọi bộ đếm trở về 0):

In [19]:
assert trace_stack(build_tree(examples[1][0]), build_tree(examples[1][1])) is True

bước  1 | T2 nút '+' dấu -1 | đẩy 2 con    | stack=3 | cnt≠0: {}
bước  2 | T2 nút 'a' dấu -1 | cnt[a] -= 1  | stack=2 | cnt≠0: {'a': -1}
bước  3 | T2 nút '+' dấu -1 | đẩy 2 con    | stack=3 | cnt≠0: {'a': -1}
bước  4 | T2 nút 'c' dấu -1 | cnt[c] -= 1  | stack=2 | cnt≠0: {'a': -1, 'c': -1}
bước  5 | T2 nút 'b' dấu -1 | cnt[b] -= 1  | stack=1 | cnt≠0: {'a': -1, 'b': -1, 'c': -1}
bước  6 | T1 nút '+' dấu +1 | đẩy 2 con    | stack=2 | cnt≠0: {'a': -1, 'b': -1, 'c': -1}
bước  7 | T1 nút '+' dấu +1 | đẩy 2 con    | stack=3 | cnt≠0: {'a': -1, 'b': -1, 'c': -1}
bước  8 | T1 nút 'c' dấu +1 | cnt[c] += 1  | stack=2 | cnt≠0: {'a': -1, 'b': -1}
bước  9 | T1 nút 'b' dấu +1 | cnt[b] += 1  | stack=1 | cnt≠0: {'a': -1}
bước 10 | T1 nút 'a' dấu +1 | cnt[a] += 1  | stack=0 | cnt≠0: {}
Kết quả: True


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** (stack + mảng 26) — `O(n)`, không phụ thuộc giới hạn đệ quy.

In [20]:
# Definition for a binary tree node.
# class Node(object):
#     def __init__(self, val=" ", left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def checkEquivalence(self, root1: 'Node', root2: 'Node') -> bool:
        cnt = [0] * 26
        stack = [(root1, 1), (root2, -1)]
        while stack:
            node, sign = stack.pop()
            if node.val == '+':
                stack.append((node.left, sign))
                stack.append((node.right, sign))
            else:
                cnt[ord(node.val) - ord('a')] += sign
        return not any(cnt)

for a1, a2, exp in examples:
    got = Solution().checkEquivalence(build_tree(a1), build_tree(a2))
    print(got, exp); assert got == exp

True True
True True
False False


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (giá trị = tổng các lá).** Với mọi cây chỉ có `+`, giá trị cây = `Σ_{lá ℓ} env[ℓ.val]`. *Chứng minh quy nạp theo chiều cao:* lá thì hiển nhiên; nút `+` có giá trị = giá trị con trái + giá trị con phải = (tổng lá bên trái) + (tổng lá bên phải) = tổng mọi lá của nó.

Từ đó giá trị cây = `Σ_v c_v · env[v]`, với `c_v` = số lá mang chữ `v`.

**Bổ đề 2 (đa thức tuyến tính).** `Σ c_v·x_v = Σ d_v·x_v` với **mọi** `x` ⇔ `c_v = d_v` với mọi `v`.
- (⇐) hiển nhiên.
- (⇒) nếu `c_u ≠ d_u`, chọn `x_u = 1`, các `x` khác `= 0` ⇒ hai vế là `c_u ≠ d_u`, mâu thuẫn.

**Thuật toán:** invariant của bộ đếm (mục 6) cho thấy khi kết thúc, `cnt[v] = c_v − d_v`. Trả `True` ⇔ mọi `cnt[v] = 0` ⇔ `c = d` ⇔ (Bổ đề 2) hai cây tương đương. ∎

Với follow-up `-`, Bổ đề 1 đổi thành: hệ số của một lá = tích các dấu trên đường từ gốc xuống (mỗi lần đi vào **con phải của nút `-`** đổi dấu). Bổ đề 2 giữ nguyên, nên Cách 5 đúng.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số nút mỗi cây (`n ≤ 4999`, lẻ), số lá `= (n+1)/2`, `h` = chiều cao cây, `Σ = 26` = số chữ cái.

| Cách | Time | Space |
|---|---|---|
| 1. One-hot 26 lần | `O(Σ·n)` | `O(h)` |
| 2. Sắp xếp lá | `O(n log n)` | `O(n)` |
| 3. Counter đệ quy | `O(n)` | `O(h + Σ)` (ngăn xếp gọi hàm) |
| 4. Stack + mảng 26 | `O(n + Σ)` | `O(h + Σ)` — stack tối đa khoảng `h + 1` phần tử mỗi cây |
| 5. Vector hệ số (+, −) | `O(n + Σ)` | `O(h + Σ)` |

Chiều cao `h` trung bình (cây "cân") là `O(log n)`, tệ nhất (cây lệch) là `(n−1)/2`. Không có khác biệt average/worst về thời gian: luôn phải đọc mọi nút. Ta đo thử thời gian trên cây lớn nhất `n = 4999`:

In [21]:
def random_tree(leaves, rng):
    """Dựng cây + ngẫu nhiên có thứ tự lá (in-order) đúng bằng `leaves`, không đệ quy."""
    nodes = [Node(v) for v in leaves]
    while len(nodes) > 1:
        i = rng.randrange(len(nodes) - 1)
        nodes[i:i + 2] = [Node("+", nodes[i], nodes[i + 1])]
    return nodes[0]

def chain_tree(leaves):
    """Cây lệch trái: (((l0 + l1) + l2) + ...) — chiều cao = số lá - 1."""
    root = Node(leaves[0])
    for v in leaves[1:]:
        root = Node("+", root, Node(v))
    return root

rng = random.Random(7)
L = [rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(2500)]
L2 = L[:]; rng.shuffle(L2)
A, B = random_tree(L, rng), chain_tree(L2)
for name, f in [("one-hot", check_bruteforce), ("sort", check_sort), ("stack", check_stack)]:
    t = time.perf_counter(); r = f(A, B); dt = time.perf_counter() - t
    print(f"{name:<8} -> {r}  ({dt*1000:.2f} ms)")

one-hot  -> True  (29.38 ms)
sort     -> True  (0.88 ms)
stack    -> True  (0.65 ms)


## 12. Bộ test

Gồm: ví dụ đề, edge case (1 nút, cùng chữ lặp lại, cây lệch sâu nhất), phản ví dụ (cùng tập chữ khác số lần), và đối chiếu **mọi cách** trên nhiều cây ngẫu nhiên có seed cố định, đúng ràng buộc (số nút lẻ, bằng nhau, `≤ 4999`).

In [22]:
ALL = {
    "bruteforce": check_bruteforce,
    "sort": check_sort,
    "recursive": lambda a, b: SolutionRecursive().checkEquivalence(a, b),
    "stack": check_stack,
    "minus-version": check_with_minus,
    "submit": lambda a, b: Solution().checkEquivalence(a, b),
}

def run_all(a, b):
    res = {k: f(a, b) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, res
    return next(iter(res.values()))

# 1) Ví dụ đề
for a1, a2, exp in examples:
    assert run_all(build_tree(a1), build_tree(a2)) == exp

# 2) Edge case
assert run_all(build_tree(["x"]), build_tree(["y"])) is False                     # 1 nút, khác chữ
assert run_all(build_tree(["+", "a", "a"]), build_tree(["+", "a", "a"])) is True   # chữ lặp
assert run_all(build_tree(mine1), build_tree(mine2)) is False                      # cùng tập, khác số lần
deep = ["abcdefghijklmnopqrstuvwxyz"[i % 26] for i in range(2500)]                  # 4999 nút
assert run_all(chain_tree(deep), chain_tree(deep[::-1])) is True
bad = deep[:]; bad[0] = "z" if bad[0] != "z" else "y"
assert run_all(chain_tree(deep), chain_tree(bad)) is False
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [23]:
# 3) Ngẫu nhiên, seed cố định: một nửa là cặp tương đương (hoán vị lá + hình dạng mới), nửa còn lại bị sửa 1 lá
rng = random.Random(20240)
cnt_true = cnt_false = 0
for _ in range(400):
    k = rng.randint(1, 60)                      # số lá  => số nút 2k-1 (lẻ, <= 4999)
    alpha = "abcdefghijklmnopqrstuvwxyz"[: rng.randint(1, 26)]
    L = [rng.choice(alpha) for _ in range(k)]
    L2 = L[:]; rng.shuffle(L2)
    if rng.random() < 0.5:
        j = rng.randrange(k); L2[j] = rng.choice(alpha)   # có thể vô tình vẫn giống
    a, b = random_tree(L, rng), random_tree(L2, rng)
    exp = Counter(L) == Counter(L2)
    got = run_all(a, b)
    assert got == exp
    cnt_true += got; cnt_false += (not got)
print("Ngẫu nhiên 400 cặp OK | True:", cnt_true, "False:", cnt_false)

Ngẫu nhiên 400 cặp OK | True: 224 False: 176


In [24]:
# 4) Ngẫu nhiên kích thước lớn (đúng giới hạn 4999 nút), chỉ các cách không đệ quy + bản nộp
rng = random.Random(99)
for _ in range(5):
    L = [rng.choice("abc") for _ in range(2500)]
    L2 = L[:]; rng.shuffle(L2)
    a, b = random_tree(L, rng), random_tree(L2, rng)
    assert check_stack(a, b) and check_sort(a, b) and Solution().checkEquivalence(a, b)
    L2[rng.randrange(2500)] = "d"
    b = random_tree(L2, rng)
    assert not check_stack(a, b) and not Solution().checkEquivalence(a, b)
print("Test lớn: OK")

Test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh **cấu trúc** cây (giống hệt nhau từng nút)

Nhiều bạn quen bài "Same Tree" nên so sánh từng nút. Nhưng đề hỏi **giá trị**, không hỏi hình dạng.

In [25]:
def wrong_same_tree(a, b):
    if a is None or b is None:
        return a is b
    return a.val == b.val and wrong_same_tree(a.left, b.left) and wrong_same_tree(a.right, b.right)

a1, a2, exp = examples[1]
print("sai :", wrong_same_tree(build_tree(a1), build_tree(a2)), "| đúng:", Solution().checkEquivalence(build_tree(a1), build_tree(a2)))

sai : False | đúng: True


**Vì sao sai:** `a + (b + c)` và `(b + c) + a` khác hình dạng nhưng bằng nhau. **Sửa:** bỏ qua cấu trúc, chỉ đếm lá (Cách 3/4).

### Lỗi 2 — So sánh **tập** chữ cái (set) thay vì multiset

In [26]:
def wrong_set(a, b):
    return set(leaves_iter(a)) == set(leaves_iter(b))

r1, r2 = build_tree(mine1), build_tree(mine2)
print(expr(r1), "vs", expr(r2))
print("sai :", wrong_set(r1, r2), "| đúng:", Solution().checkEquivalence(r1, r2))

(a + (a + b)) vs ((b + b) + a)
sai : True | đúng: False


**Vì sao sai:** `set` làm mất thông tin **số lần**. `2a + b ≠ a + 2b` (thử `a=1, b=0`). **Sửa:** dùng `Counter`/mảng đếm, hoặc so sánh `sorted(...)`.

### Lỗi 3 — Kiểm tra `sum(cnt) == 0` thay vì **mọi** phần tử bằng 0

In [27]:
def wrong_sum(a, b):
    cnt = [0] * 26
    st = [(a, 1), (b, -1)]
    while st:
        x, s = st.pop()
        if x.val == "+":
            st.append((x.left, s)); st.append((x.right, s))
        else:
            cnt[ord(x.val) - 97] += s
    return sum(cnt) == 0          # SAI

a1, a2, exp = examples[2]
print("sai :", wrong_sum(build_tree(a1), build_tree(a2)), "| đúng:", Solution().checkEquivalence(build_tree(a1), build_tree(a2)))

sai : True | đúng: False


**Vì sao sai:** hai cây có cùng số lá nên tổng `cnt` **luôn** là 0! Ở Ví dụ 3, `cnt[c] = +1`, `cnt[d] = −1`, tổng 0 nhưng không tương đương. **Sửa:** `not any(cnt)` hoặc `all(v == 0 for v in cnt)`.

### Lỗi 4 — So sánh chuỗi biểu thức in-order

In [28]:
def wrong_inorder(a, b):
    return "".join(leaves_iter(a)) == "".join(leaves_iter(b))

a1, a2, exp = examples[1]
print("in-order:", "".join(leaves_iter(build_tree(a1))), "vs", "".join(leaves_iter(build_tree(a2))))
print("sai :", wrong_inorder(build_tree(a1), build_tree(a2)), "| đúng:", exp)

in-order: abc vs bca
sai : False | đúng: True


**Vì sao sai:** thứ tự lá thay đổi khi đổi chỗ hai con (giao hoán). **Sửa:** so sánh sau khi **sắp xếp** hoặc đếm.

### Lỗi 5 — Đệ quy không tăng giới hạn trên cây lệch sâu

Cây lệch 4999 nút có độ sâu 2500 > 1000. Ta bắt `RecursionError` để chứng minh (không để lỗi văng ra):

In [29]:
def wrong_recursive_no_limit(a, b):
    cnt = Counter()
    def dfs(x, s):
        if x.val != "+":
            cnt[x.val] += s; return
        dfs(x.left, s); dfs(x.right, s)
    dfs(a, 1); dfs(b, -1)
    return not any(cnt.values())

old = sys.getrecursionlimit(); sys.setrecursionlimit(1000)
big = chain_tree(deep)
try:
    wrong_recursive_no_limit(big, big)
    print("không lỗi (máy này có giới hạn lớn)")
except RecursionError as e:
    print("Bắt được RecursionError:", type(e).__name__)
finally:
    sys.setrecursionlimit(max(old, 10000))
print("Bản stack vẫn chạy:", Solution().checkEquivalence(big, big))

Bắt được RecursionError: RecursionError
Bản stack vẫn chạy: True


**Sửa:** dùng stack (Cách 4) hoặc `sys.setrecursionlimit(10000)` (Cách 3).

### Lỗi 6 — Dùng lại cách đếm `±1` cho follow-up có phép `-`

In [30]:
p = build_tree(["-", "a", "b"])   # a - b
q = build_tree(["-", "b", "a"])   # b - a
print(expr(p), "vs", expr(q))
print("đếm lá (sai cho '-'):", check_sort(p, q), "| vector hệ số (đúng):", check_with_minus(p, q))

(a - b) vs (b - a)
đếm lá (sai cho '-'): True | vector hệ số (đúng): False


**Vì sao sai:** phép trừ **không giao hoán**: `a − b ≠ b − a`. Cùng túi lá nhưng hệ số khác dấu. **Sửa:** truyền dấu xuống con phải của nút `-` (Cách 5).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**tương đương với mọi giá trị biến**" + toán tử **giao hoán, kết hợp** (`+`, `*`, `min`, `max`, `xor`, …) ⇒ nghĩ ngay tới **so sánh multiset lá**.
- "Hai tập có cùng phần tử kể cả số lần" ⇒ pattern **một bộ đếm cộng/trừ** (giống bài Valid Anagram): `+1` cho bên A, `−1` cho bên B, kiểm tra toàn 0.
- Bảng chữ nhỏ (26 chữ) ⇒ dùng `[0]*26` thay `dict`.
- Cây có thể rất sâu trong Python ⇒ ưu tiên **stack** hoặc tăng `recursionlimit`.
- Nhớ kiểm tra "**mọi** phần tử = 0", không phải "tổng = 0".
- Khi có phép không giao hoán (`-`), nghĩ tới **dấu tích lũy dọc đường đi** (hệ số của lá).

## 15. Biến thể của bài

1. **Follow-up `-`**: vector hệ số / dấu tích lũy (Cách 5).
2. **Có thêm phép `*`**: biểu thức thành **đa thức** (nhiều số hạng là tích các biến). Phải chuẩn hóa thành dict `{tuple biến đã sắp xếp: hệ số}` rồi so sánh; hoặc dùng kiểm tra xác suất (gán ngẫu nhiên modulo số nguyên tố lớn — bổ đề Schwartz–Zippel).
3. **Lá là hằng số** (số nguyên) thay vì biến: chỉ cần tính giá trị hai cây và so sánh.
4. **Nhóm k cây tương đương** thành các lớp: dùng `tuple(cnt)` làm khóa của `dict`.
5. **LeetCode 1628 — Design an Expression Tree With Evaluate Function**: dựng cây từ postfix và tính giá trị.
6. **LeetCode 100 / 951 — Same Tree / Flip Equivalent Binary Trees**: so sánh cấu trúc (có/không cho lật con).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. One-hot | Gán từng biến = 1, so giá trị | `O(26·n)` | `O(h)` | Bám định nghĩa, kiểm chứng; mở rộng thành kiểm tra ngẫu nhiên khi có `*` |
| 2. Sắp xếp lá | So hai list lá đã sort | `O(n log n)` | `O(n)` | Nhanh nghĩ ra, đủ qua với `n ≤ 4999` |
| 3. Counter đệ quy | `+1`/`−1` khi gặp lá | `O(n)` | `O(h)` | Code ngắn nhất; nhớ tăng recursionlimit |
| 4. Stack + mảng 26 | Như 3 nhưng không đệ quy | `O(n)` | `O(h)` | **Bản nộp khuyên dùng** |
| 5. Dấu tích lũy | Hệ số ±1 theo đường đi | `O(n)` | `O(h)` | Follow-up có phép `-` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Cho các cặp cây (mảng level-order), đoán `checkEquivalence`:
(a) `[+,a,b]` vs `[+,b,a]`; (b) `[+,a,+,null,null,a,a]` vs `[+,+,a,a,a]`; (c) `[+,a,b]` vs `[+,a,a]`; (d) `[z]` vs `[z]`.

**Bài 2 (tính tay):** Với `root1 = [+,+,+,a,b,c,a]` và `root2 = [+,b,+,null,null,a,c]`, hãy viết biểu thức của mỗi cây, tính bộ đếm `cnt` (cây 1 cộng, cây 2 trừ) và kết luận. (Lưu ý: hai cây này có số nút khác nhau, nên không thỏa ràng buộc đề — bài tập chỉ để luyện đếm.)

**Bài 3 (điền TODO):** Hoàn thiện `leaf_count(root)` trả về `Counter` số lần xuất hiện của mỗi chữ ở lá, duyệt bằng stack.

**Bài 4 (tự viết lại):** Viết `check_bfs(root1, root2)` dùng **BFS** (`deque`) thay vì stack, với một bộ đếm mảng 26.

**Bài 5 (mini challenge):** Cho danh sách `k` cây (có cả `+` và `-`), viết `group_equivalent(roots)` trả về danh sách các nhóm chỉ số cây tương đương (mỗi nhóm sắp xếp tăng, các nhóm sắp theo phần tử đầu).

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def leaf_count(root):
    cnt = Counter()
    st = [root]
    # TODO: lấy nút ra; nếu là '+' thì đẩy 2 con, ngược lại cnt[x.val] += 1
    return cnt

In [32]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def check_bfs(root1, root2):
    # TODO: deque chứa (node, sign); popleft; đếm vào mảng 26
    pass

def group_equivalent(roots):
    # TODO: tính vector hệ số của từng cây -> dùng làm khóa dict
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: viết biểu thức của từng cây. Mức 2: đếm túi lá. Mức 3: (a) `True`; (b) `True` (`3a` cả hai); (c) `False`; (d) `True`.

**Bài 2.** Mức 1: cây 1 là `((a + b) + (c + a))`, cây 2 là `b + (a + c)`. Mức 2: túi 1 `{a:2,b:1,c:1}`, túi 2 `{a:1,b:1,c:1}`. Mức 3: `cnt = {a:1}` ≠ 0 ⇒ `False`.

**Bài 3.** Mức 1: vòng `while st:`. Mức 2: `x = st.pop()`, kiểm tra `x.val == '+'`. Mức 3: đẩy `x.left`, `x.right`; lá thì `cnt[x.val] += 1`.

**Bài 4.** Mức 1: `q = deque([(root1, 1), (root2, -1)])`. Mức 2: `x, s = q.popleft()`. Mức 3: phần còn lại giống hệt Cách 4 — BFS hay DFS không quan trọng vì chỉ cần thăm mọi lá đúng một lần.

**Bài 5.** Mức 1: dùng `coef_vector` (Cách 5). Mức 2: khóa = `tuple(sorted((k, v) for k, v in vec.items() if v))`. Mức 3: `dict` khóa → list chỉ số, rồi `sorted(groups.values())`.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
pairs = [(["+", "a", "b"], ["+", "b", "a"]),
         (["+", "a", "+", None, None, "a", "a"], ["+", "+", "a", "a", "a"]),
         (["+", "a", "b"], ["+", "a", "a"]),
         (["z"], ["z"])]
b1 = [Solution().checkEquivalence(build_tree(x), build_tree(y)) for x, y in pairs]
print("Bài 1:", b1)
assert b1 == [True, True, False, True]

# Bài 2
r1 = build_tree(["+", "+", "+", "a", "b", "c", "a"]); r2 = build_tree(["+", "b", "+", None, None, "a", "c"])
print("Bài 2:", expr(r1), "|", expr(r2))
c = Counter(leaves_iter(r1)); c.subtract(Counter(leaves_iter(r2)))
print("   cnt khác 0:", {k: v for k, v in c.items() if v})
assert {k: v for k, v in c.items() if v} == {"a": 1}

Bài 1: [True, True, False, True]
Bài 2: ((a + b) + (c + a)) | (b + (a + c))
   cnt khác 0: {'a': 1}


In [34]:
# Bài 3, 4 — đáp án
def leaf_count(root):
    cnt = Counter()
    st = [root]
    while st:
        x = st.pop()
        if x.val == "+":
            st.append(x.left); st.append(x.right)
        else:
            cnt[x.val] += 1
    return cnt

def check_bfs(root1, root2):
    cnt = [0] * 26
    q = deque([(root1, 1), (root2, -1)])
    while q:
        x, s = q.popleft()
        if x.val == "+":
            q.append((x.left, s)); q.append((x.right, s))
        else:
            cnt[ord(x.val) - 97] += s
    return not any(cnt)

assert leaf_count(build_tree(mine1)) == Counter({"a": 2, "b": 1})
rng = random.Random(5)
for _ in range(200):
    k = rng.randint(1, 40)
    L = [rng.choice("abcde") for _ in range(k)]
    L2 = L[:]; rng.shuffle(L2)
    if rng.random() < 0.5:
        L2[rng.randrange(k)] = rng.choice("abcde")
    a, b = random_tree(L, rng), random_tree(L2, rng)
    assert leaf_count(a) == Counter(L)
    assert check_bfs(a, b) == Solution().checkEquivalence(a, b)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [35]:
# Bài 5 — đáp án
def group_equivalent(roots):
    groups = {}
    for i, r in enumerate(roots):
        vec = coef_vector(r)
        key = tuple(sorted((k, v) for k, v in vec.items() if v))
        groups.setdefault(key, []).append(i)
    return sorted(groups.values())

trees = [build_tree(["-", "a", "-", None, None, "b", "c"]),   # a - b + c
         build_tree(["+", "a", "b"]),                          # a + b
         build_tree(["+", "-", "c", "a", "b"]),                # a - b + c
         build_tree(["+", "b", "a"]),                          # a + b
         build_tree(["-", "a", "b"])]                          # a - b
res = group_equivalent(trees)
print("Bài 5:", res)
assert res == [[0, 2], [1, 3], [4]]

# kiểm chứng bằng gán ngẫu nhiên
rng = random.Random(11)
for g in res:
    for e in ({ch: rng.randint(-9, 9) for ch in "abc"} for _ in range(50)):
        assert len({evaluate_pm(trees[i], e) for i in g}) == 1
print("Kiểm chứng ngẫu nhiên: OK")

Bài 5: [[0, 2], [1, 3], [4]]
Kiểm chứng ngẫu nhiên: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "so sánh multiset lá bằng một bộ đếm":**

```python
cnt = [0] * 26
stack = [(root1, 1), (root2, -1)]
while stack:
    node, sign = stack.pop()
    if node.val == '+':                     # (follow-up) elif node.val == '-': con phải nhận -sign
        stack.append((node.left, sign))
        stack.append((node.right, sign))
    else:
        cnt[ord(node.val) - ord('a')] += sign
return not any(cnt)
```

**Checklist:**
- [ ] Đếm **số lần** từng chữ (multiset), không dùng `set`, không so cấu trúc?
- [ ] Kiểm tra **mọi** `cnt[v] == 0` (không phải `sum(cnt) == 0`)?
- [ ] Bỏ qua nút `'+'` khi đếm, chỉ đếm lá?
- [ ] Cây sâu tới ~2500: dùng stack hoặc `sys.setrecursionlimit`?
- [ ] Trường hợp 1 nút (`[x]`) hoạt động?
- [ ] Nếu có `'-'`: đổi dấu ở con **phải** của nút `'-'` (không phải con trái)?
- [ ] Đã test ví dụ đề + cùng tập khác số lần + cây lệch sâu + ngẫu nhiên đối chiếu?